In [6]:
# -------------------------------------------------------------
# Leica LAS AF ViewerScaling Extractor (Jupyter + .py friendly)
# -------------------------------------------------------------
import xml.etree.ElementTree as ET
from pathlib import Path
import json
import re
import sys

# Data folder (always relative to project root)
DATA_DIR = Path("source") / "10" / "MetaData"
OUTPUT_DIR = Path(".")  # JSONs saved at project root

# ── 2. Core functions ──
def sanitize_filename(name: str) -> str:
    """Make string safe for filenames"""
    return re.sub(r'[<>:"/\\|?*]', '_', str(name).strip())

def extract_channel_scaling(xml_path: Path):
    """Extract Gray/Red channel scaling from Leica XML metadata"""
    tree = ET.parse(xml_path)
    root = tree.getroot()

    # Get image name
    name_elem = root.find(".//Name")
    image_name = name_elem.text.strip() if name_elem is not None and name_elem.text else xml_path.stem

    # Find ViewerScaling
    viewer = root.find(".//Attachment[@Name='ViewerScaling']")
    if viewer is None:
        raise ValueError("ViewerScaling attachment not found in XML")

    scaling_info = {}
    intensity_info = {}

    for i, ch in enumerate(viewer.findall("ChannelScalingInfo")):
        channel = "Gray" if i == 0 else "Red"
        lut = ch.get("BackgroundLutName", "") or "(none)"
        white = float(ch.get("WhiteValue", 0))
        black = float(ch.get("BlackValue", 0))
        gamma = float(ch.get("GammaValue", 1))
        auto = ch.get("Automatic", "0") == "1"

        scaling_info[channel] = {
            "LUT": lut,
            "WhiteValue": white,
            "BlackValue": black,
            "Gamma": gamma,
            "Automatic": auto
        }

        w12 = round(white * 4095, 2)
        b12 = round(black * 4095, 2)
        intensity_info[channel] = {
            "WhiteLevel_12bit": w12,
            "BlackLevel_12bit": b12,
            "DynamicRange_12bit": round(w12 - b12, 2)
        }

    return image_name, scaling_info, intensity_info

def save_scaling_to_json(xml_path: Path, base_name: str = "channel_scaling.json"):
    """Extract scaling and save as pretty JSON with image name in filename"""
    image_name, scaling, intensity = extract_channel_scaling(xml_path)
    safe_name = sanitize_filename(image_name)
    output_path = OUTPUT_DIR / f"{safe_name}_{base_name}"

    data = {
        "ImageName": image_name,
        "SourceXML": xml_path.name,
        "Scaling_Normalized": scaling,
        "Intensity_12bit": intensity
    }

    output_path.write_text(json.dumps(data, indent=4), encoding="utf-8")
    print(f"Saved → {output_path}")
    return output_path

# ── 3. Run extraction ──
xml_file = DATA_DIR / "10 P 1.xml"

print(f"Looking for:  {xml_file}\n")

if not xml_file.exists():
    print("XML file not found!")
    print(f"   Expected location: {xml_file}")
else:
    try:
        img_name, scaling, intensity = extract_channel_scaling(xml_file)
        print(f"Loading image: {img_name}\n")
        print(json.dumps(scaling, indent=2))
        save_scaling_to_json(xml_file, "channel_scaling.json")
    except Exception as e:
        print(f"Error: {e}")

Looking for:  source/10/MetaData/10 P 1.xml

Loading image: 10 P 1

{
  "Gray": {
    "LUT": "Gray",
    "WhiteValue": 0.188034188034188,
    "BlackValue": 0.043956043956044,
    "Gamma": 1.0,
    "Automatic": false
  },
  "Red": {
    "LUT": "(none)",
    "WhiteValue": 0.0732600732600733,
    "BlackValue": 0.043956043956044,
    "Gamma": 1.0,
    "Automatic": false
  }
}
Saved → 10 P 1_channel_scaling.json
